### Import Libs

In [1]:
!pip -q install torch sentencepiece records babel tqdm tabulate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00


In [2]:
import json
from pathlib import Path
import sentencepiece as spm
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


### Download Data from GitHub

In [3]:
!git clone https://github.com/salesforce/WikiSQL

Cloning into 'WikiSQL'...
remote: Enumerating objects: 389, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 389 (delta 186), reused 154 (delta 154), pack-reused 194 (from 1)
Receiving objects: 100% (389/389), 50.72 MiB | 41.75 MiB/s, done.
Resolving deltas: 100% (213/213), done.


In [4]:
%cd WikiSQL
!ls


/content/WikiSQL
annotate.py  CONTRIBUTING-ARCHIVED.md  lib	  requirements.txt
CODEOWNERS   data.tar.bz2	       LICENSE	  test
collection   evaluate.py	       README.md  version.txt


In [5]:
!tar xvjf data.tar.bz2
%cd ..

data/
data/train.jsonl
data/test.tables.jsonl
data/test.db
data/dev.tables.jsonl
data/dev.db
data/test.jsonl
data/train.tables.jsonl
data/train.db
data/dev.jsonl
/content


### Load & Preprocess

In [6]:
DATA_DIR = Path("WikiSQL/data")
AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<"]
MAX_COLS = 64  # column tokens <c0> ... <c63>


def load_split(split):
    """Return (examples, tables) for 'train', 'dev' or 'test'."""
    tables = {}

    with open(DATA_DIR / f"{split}.tables.jsonl", encoding="utf-8") as f:
        for line in f:
            t = json.loads(line)
            tables[t["id"]] = t

    with open(DATA_DIR / f"{split}.jsonl", encoding="utf-8") as f:
        examples = [json.loads(line) for line in f]

    return examples, tables


def encode_source(question, header):
    """question <sep> <c0> col name <c1> col name ..."""
    cols = " ".join(
        f"<c{i}> {name}" for i, name in enumerate(header)
    )
    return f"{question.strip()} <sep> {cols}".lower()


def encode_target(sql):
    """{'sel','agg','conds'} -> 'select count <c3> where <c1> = kim manners'"""
    out = ["select"]

    if sql["agg"]:
        out.append(AGG_OPS[sql["agg"]].lower())

    out.append(f"<c{sql['sel']}>")

    for i, (col, op, val) in enumerate(sql["conds"]):
        out += [
            "where" if i == 0 else "and",
            f"<c{col}>",
            COND_OPS[op],
            str(val)
        ]

    return " ".join(out).lower()


def build_pairs(split):
    examples, tables = load_split(split)
    pairs = []

    for ex in examples:
        header = tables[ex["table_id"]]["header"]

        pairs.append({
            "table_id": ex["table_id"],
            "src": encode_source(ex["question"], header),
            "tgt": encode_target(ex["sql"]),
        })

    return pairs

In [7]:
for split in ["train", "dev", "test"]:
        pairs = build_pairs(split)

        with open(
            f"{split}_pairs.jsonl", "w", encoding="utf-8"
        ) as f:
            for p in pairs:
                f.write(json.dumps(p, ensure_ascii=False) + "\n")

        print(f"{split}: {len(pairs)} pairs")
        print(pairs[0]["src"])
        print(pairs[0]["tgt"])

train: 56355 pairs
tell me what the notes are for south australia <sep> <c0> state/territory <c1> text/background colour <c2> format <c3> current slogan <c4> current series <c5> notes
select <c5> where <c3> = south australia
dev: 8421 pairs
what position does the player who played for butler cc (ks) play? <sep> <c0> player <c1> no. <c2> nationality <c3> position <c4> years in toronto <c5> school/club team
select <c3> where <c5> = butler cc (ks)
test: 15878 pairs
what is terrence ross' nationality <sep> <c0> player <c1> no. <c2> nationality <c3> position <c4> years in toronto <c5> school/club team
select <c2> where <c0> = terrence ross


### Tokenizer

In [8]:
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPECIAL = ["<sep>"] + [f"<c{i}>" for i in range(MAX_COLS)]


def read_pairs(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def train_tokenizer(train_path="train_pairs.jsonl", vocab_size=8000):
    pairs = read_pairs(train_path)

    with open("spm_corpus.txt", "w", encoding="utf-8") as f:
        for p in pairs:  # ONE shared vocabulary for
            f.write(p["src"] + "\n")  # source and target, as in
            f.write(p["tgt"] + "\n")  # the original Transformer

    spm.SentencePieceTrainer.train(
        input="spm_corpus.txt",
        model_prefix="sql_sp",
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=1.0,
        user_defined_symbols=SPECIAL,  # never split <sep>, <c0>, ...
        pad_id=PAD_ID,
        unk_id=UNK_ID,
        bos_id=BOS_ID,
        eos_id=EOS_ID,
    )

    return spm.SentencePieceProcessor(model_file="sql_sp.model")


In [9]:
sp = train_tokenizer()
p = read_pairs("dev_pairs.jsonl")[0]

print(sp.encode(p["src"], out_type=str))
print(sp.encode(p["tgt"], out_type=str))
print(sp.decode(sp.encode(p["tgt"])) == p["tgt"])

['▁what', '▁position', '▁does', '▁the', '▁player', '▁who', '▁played', '▁for', '▁but', 'ler', '▁cc', '▁(', 'ks', ')', '▁play', '?', '▁', '<sep>', '▁', '<c0>', '▁player', '▁', '<c1>', '▁no', '.', '▁', '<c2>', '▁nationality', '▁', '<c3>', '▁position', '▁', '<c4>', '▁years', '▁in', '▁toronto', '▁', '<c5>', '▁school', '/', 'club', '▁team']
['▁select', '▁', '<c3>', '▁where', '▁', '<c5>', '▁=', '▁but', 'ler', '▁cc', '▁(', 'ks', ')']
True


### Dataloaders

In [ ]:
class SQLDataset(Dataset):
    """
    Each item: (source ids, target ids). Target = <s> ... </s>.
    train=True drops over-long pairs.
    train=False keeps EVERY row in file order (needed for evaluation).
    """

    def __init__(self, path, sp, train=True, max_src=160, max_tgt=64):
        self.items, skipped = [], 0

        for p in read_pairs(path):
            src = sp.encode(p["src"]) + [EOS_ID]
            tgt = [BOS_ID] + sp.encode(p["tgt"]) + [EOS_ID]

            if train and (len(src) > max_src or len(tgt) > max_tgt):
                skipped += 1
                continue

            self.items.append((src, tgt))

        print(f"{path}: kept {len(self.items)}, skipped {skipped}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    srcs, tgts = zip(*batch)

    S, T = max(map(len, srcs)), max(map(len, tgts))

    src = torch.full(
        (len(batch), S),
        PAD_ID,
        dtype=torch.long
    )

    tgt = torch.full(
        (len(batch), T),
        PAD_ID,
        dtype=torch.long
    )

    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)


    return src, tgt  # (B, S), (B, T)


def make_loader(path, sp, train, batch_size=64):
    return DataLoader(
        SQLDataset(path, sp, train=train),
        batch_size=batch_size,
        shuffle=train,
        collate_fn=collate
    )

### Embeddings & Positional encoding

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Section 3.4: embeddings are multiplied by sqrt(d_model).
    One instance is shared by the encoder, the decoder and
    (optionally) the final output projection.
    """

    def __init__(self, vocab_size, d_model, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.scale = math.sqrt(d_model)

    def forward(self, ids):  # (Batch size, seq lenght) -> (Batch size, seq lenght, d_model)
        return self.emb(ids) * self.scale


class PositionalEncoding(nn.Module):
    """
    Section 3.5: fixed sinusoidal encodings.
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pos = torch.arange(max_len).unsqueeze(1)  # (L, 1)

        div = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )  # (d/2,)

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )  # (1, L, d)

    def forward(self, x):  # x: (B, L, d_model)
        return self.dropout(
            x + self.pe[:, :x.size(1)]
        )


class InputLayer(nn.Module):
    """
    token ids -> scaled embedding + positional encoding -> dropout
    """

    def __init__(
        self,
        token_emb,
        d_model,
        max_len=512,
        dropout=0.1
    ):
        super().__init__()

        self.tok = token_emb
        self.pos = PositionalEncoding(
            d_model,
            max_len,
            dropout
        )

    def forward(self, ids):
        return self.pos(self.tok(ids))

### Check Starter Codes

In [12]:
sp = spm.SentencePieceProcessor(model_file="sql_sp.model")

train_dl = make_loader("train_pairs.jsonl", sp, train=True)
dev_dl = make_loader("dev_pairs.jsonl", sp, train=False)


src, tgt = next(iter(train_dl))

print("src", tuple(src.shape), "tgt", tuple(tgt.shape))


d_model = 256

shared = TokenEmbedding(
    sp.get_piece_size(),
    d_model,
    PAD_ID
)

enc_in = InputLayer(shared, d_model)  # encoder input
dec_in = InputLayer(shared, d_model)  # decoder input (same weights)


x = enc_in(src)  # (B, S, 256) -> goes into YOUR encoder
y = dec_in(tgt[:, :-1])  # (B, T-1, 256) -> goes into YOUR decoder

print(
    "encoder input",
    tuple(x.shape),
    "decoder input",
    tuple(y.shape)
)

train_pairs.jsonl: kept 56336, skipped 19
dev_pairs.jsonl: kept 8421, skipped 0
src (64, 130) tgt (64, 38)
encoder input (64, 130, 256) decoder input (64, 37, 256)


### Achitecture
Layers:
* Multi-head Attention
* Norm & Skip connection
* FFN

Encoder:
* Encoder_layer (self-attention → add & norm → FFN → add & norm)
* Encoder (N layers stacked)

Decoder:
* Decoder_layer (masked self-attention →add & norm →cross-attention over the encoder output → add & norm → FFN → add & norm)
* Decoder (N layers Stacked)

In [ ]:
import math
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """multi-head attention mechanism
    """

    def __init__(self, d_model: int, h: int, dropout: float=0.1):
        """initializer of multi-head attention mechanism

        Args:
            d_model (int): model hidden dimension
            h (int): number of heads
            dropout (float, optional): dropout to be applied on attention scores. Defaults to 0.1.
        """
        super().__init__()
        
        self.d_model = d_model # embedding dim
        self.h = h # no. heads
        self.d_k = d_model // h # dim of each head
        
        self.dropout = nn.Dropout(dropout)
        
        self.w_k = nn.Linear(d_model, d_model) # key
        self.w_q = nn.Linear(d_model, d_model) # q
        self.w_v = nn.Linear(d_model, d_model) # value
        
        self.w_o = nn.Linear(d_model, d_model) # output
        

    def forward(self, query, key, value, mask):
        """forward pass for the multi-head attention mechanism

        Args:
            query (tensor): query tensor of shape (batch, seq_len, d_model)
            key (tensor): key tensor of shape (batch, seq_len, d_model)
            value (tensor): value tensor of shape (batch, seq_len, d_model)
            mask (tensor): mask tensor of shape (___)

        Returns:
            tensor: output tensor of shape (batch, seq_len, d_model)
        """
        
        Q = self.w_q(query) # (batch, seq_len, d_model) @ (d_model, d_model) -> (batch, seq_len, d_model)
        K = self.w_k(key)   # //
        V = self.w_v(value) # //
        
        q = Q.view(Q.size(0), Q.size(1), self.h, self.d_k) # (batch , seq_len, h, d_k)
        k = K.view(K.size(0), K.size(1), self.h, self.d_k) # //
        v = V.view(V.size(0), V.size(1), self.h, self.d_k) # //
        
        q = q.transpose(1, 2) # (batch, h, seq_len, d_k)
        k = k.transpose(1, 2) # //
        v = v.transpose(1, 2) # //
        
        att = q @ k.transpose(-2, -1) / math.sqrt(self.d_k) # (batch, h ,seq_len, d_k) @ (batch, h, d_k, seq_len) -> (batch, h, seq_len, seq_len)
        
        if mask is not None:
            att = att.masked_fill(mask == 0, float("-inf"))
        
        score = torch.softmax(att, dim=-1)
        
        if self.dropout is not None:
            score = self.dropout(score)
        
        score = score @ v # (batch, h, seq_len, seq_len) @ (batch, h, seq_len, d_k) -> (batch, h, seq_len, d_k)
        score = score.transpose(1, 2).contiguous().view(score.size(0), -1, self.d_model) # (batch, seq_len, d_model)
        
        return self.w_o(score) # (batch, seq_len, d_model) @ (d_model, d_model) -> (batch, seq_len, d_model)
        

In [ ]:
class AddNorm(nn.Module):
    """Add & Norm layer acts as a residual connection followed by layer normalization"""
    def __init__(self, d_model: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, sublayer_x):
        added = x + sublayer_x # (batch, seq_len, d_model) + (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        output = self.norm(added)
        
        return output # (batch, seq_len, d_model)

class FeedForward(nn.Module):
    """2 layer point wise feed forward network with ReLU activation and dropout"""
    def __init__(self, d_model: int, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        # x shape: (Batch, Seq_Len, d_model)
        
        x = self.linear1(x) # (Batch, seq_len, d_model) @ (d_model, d_ff) -> (Batch, Seq_Len, d_ff)
        x = self.relu(x)   # (Batch, Seq_Len, d_ff)
        x = self.dropout(x)       
        output = self.linear2(x) # (Batch, Seq_Len, d_ff) @ (d_ff, d_model) -> (Batch, Seq_Len, d_model)
        
        return output

In [ ]:
class EncoderLayer(nn.Module):
    """encoder layer module
    x -> multi-head attention -> residual connection -> feed forward network -> residual connection
    """

    def __init__(self, multi_head_attention: MultiHeadAttention, feed_forward_network: FeedForward, add_norm: AddNorm, dropout: float=0.1):
        """initializer for encoder layer module

        Args:
            multi_head_attention (MultiHeadAttention): multi-head attention module
            feed_forward_network (FeedForwardNetwork): feed forward network module
            add_norm (AddNorm): add & norm module
            d_model (int): model hidden dimention
            dropout (float, optional): dropout to be applied on encoder layer. Defaults to 0.1.
        """
        super().__init__()
        self.mha = multi_head_attention
        self.ffn = feed_forward_network
        self.add_norm = add_norm
        self.add_norm2 = AddNorm(multi_head_attention.d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, src_mask):
        """forward pass for encoder layer module

        Args:
            x (tensor): input tensor of shape (batch, s_seq_len, d_model)
            seq_mask (tensor): sequence mask tensor of shape (___)

        Returns:
            tensor: output tensor of shape (batch, s_seq_len, d_model)
        """
        a = self.mha(x, x, x, src_mask) # (batch, s_seq_len, d_model)
        a = self.add_norm(x, a) # (batch, s_seq_len, d_model)
        
        f = self.ffn(a) # (batch, s_seq_len, d_model)
        output = self.add_norm2(a, f) # (batch, s_seq_len, d_model)
        
        return output

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d_model: int, N: int , dropout: float = 0.1):
        """Stackes N layers of EncoderLayer to form the complete encoder module"""
        
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(MultiHeadAttention(d_model, h=8, dropout=dropout)
                                                  , FeedForward(d_model, d_ff=1024, dropout=dropout)
                                                  , AddNorm(d_model),
                                                  dropout) for _ in range(N)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, src_mask):
        
        for layer in self.layers:
            x = layer(x, src_mask)

        output = self.norm(x)
        return output

In [ ]:
class DecoderLayer(nn.Module):
    """ Decoder layer module
    y -> masked multi-head attention -> add norm -> multi-head cross attention -> add norm -> FFN -> residual connection
    """
    def __init__(self, multi_head_attention: MultiHeadAttention, feed_forward_network: FeedForward, add_norm: AddNorm, dropout: float=0.1):
        """initializer for decoder layer module

        Args:
            multi_head_attention (MultiHeadAttention): multi-head attention module
            feed_forward_network (FeedForwardNetwork): feed forward network module
            add_norm (AddNorm): add & norm module
            dropout (float): Defaults to 0.1.
        """
        super().__init__()
        self.mha1 = multi_head_attention
        self.mha2 = MultiHeadAttention(
            multi_head_attention.d_model,
            multi_head_attention.h,
            dropout
        )
        self.ffn = feed_forward_network
        self.add_norm1 = add_norm
        self.add_norm2 = AddNorm(multi_head_attention.d_model)
        self.add_norm3 = AddNorm(multi_head_attention.d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, enc_output, src_mask, tgt_mask):
        """forward pass for decoder layer module

        Args:
            x (tensor): input tensor of shape (batch, t_seq_len, d_model)
            enc_output (tensor): encoder output tensor of shape (batch, s_seq_len, d_model)
            src_mask (tensor): source mask tensor of shape (batch, 1, 1, s_seq_len)
            tgt_mask (tensor): target mask tensor of shape (batch, 1, t_seq_len, t_seq_len)
        """
        a = self.mha1(x, x, x, tgt_mask) # (batch, t_seq_len, d_model)
        b = self.add_norm1(x, a) 
        
        c = self.mha2(b, enc_output, enc_output, src_mask)
        d = self.add_norm2(b, c) 

        e = self.ffn(d) 
        out = self.add_norm3(d, e) # (batch, t_seq_len, d_model)
        return out

In [ ]:
class Decoder(nn.Module):
    """Stack N decoder layers and apply a final layer normalization."""

    def __init__(self, d_model: int, N: int, h: int = 8,
                 d_ff: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(
                MultiHeadAttention(d_model, h=h, dropout=dropout),
                FeedForward(d_model, d_ff=d_ff, dropout=dropout),
                AddNorm(d_model),
                dropout
            )
            for _ in range(N)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        return self.norm(x)
